# CardioIA — Fase 2, Parte 1
## Extração de sintomas e sugestão de diagnóstico (mapa de conhecimento)

Este notebook lê as frases simuladas de pacientes e o CSV de ontologia (`sintoma_1`, `sintoma_2`, `doenca_associada`), pontua as regras e sugere o diagnóstico mais compatível.

**Google Colab:** após abrir este `.ipynb` no Colab, uma opção é **clonar o repositório** (`!git clone ...`) ou **enviar** `mapa_conhecimento.csv` e `sintomas_pacientes.txt` para o ambiente e ajustar `DATA` na célula seguinte para a pasta onde estiverem os arquivos (por exemplo `/content/datasets`).

In [ ]:
from pathlib import Path

import pandas as pd

# Ajuste se necessário: no clone do GitHub costuma ser Path("../datasets") a partir de notebooks/
candidatos = [
    Path("datasets"),
    Path("../datasets"),
    Path("/content/datasets"),
]
DATA = next((p for p in candidatos if (p / "mapa_conhecimento.csv").exists()), Path("datasets"))
print("Pasta de dados:", DATA.resolve())

In [ ]:
def sintoma_na_frase(sintoma: str, texto: str) -> bool:
    s = sintoma.lower().strip()
    t = texto.lower()
    if s in t:
        return True
    if s == "desmaio" and "desmai" in t:
        return True
    return False


def pontuar_linha(frase: str, sintoma_1: str, sintoma_2: str) -> int:
    m1 = sintoma_na_frase(sintoma_1, frase)
    m2 = sintoma_na_frase(sintoma_2, frase)
    if m1 and m2:
        return 2
    if m1 or m2:
        return 1
    return 0


def sugerir_diagnosticos(frase: str, mapa: pd.DataFrame) -> str:
    scores: dict[str, int] = {}
    pares_completos: dict[str, int] = {}
    for _, row in mapa.iterrows():
        p = pontuar_linha(frase, str(row["sintoma_1"]), str(row["sintoma_2"]))
        if p == 0:
            continue
        doenca = str(row["doenca_associada"])
        scores[doenca] = scores.get(doenca, 0) + p
        if p == 2:
            pares_completos[doenca] = pares_completos.get(doenca, 0) + 1
    if not scores:
        return "Nenhuma associação encontrada no mapa de conhecimento."
    ordenado = sorted(
        scores.items(),
        key=lambda x: (-pares_completos.get(x[0], 0), -x[1], x[0]),
    )
    principal = ordenado[0][0]
    detalhe = "; ".join(f"{d} ({v})" for d, v in ordenado[:5])
    return f"Sugestão principal: {principal}. Agregado: {detalhe}"

In [ ]:
mapa = pd.read_csv(DATA / "mapa_conhecimento.csv", encoding="utf-8")
with open(DATA / "sintomas_pacientes.txt", encoding="utf-8") as f:
    frases = [ln.strip() for ln in f if ln.strip()]

for i, frase in enumerate(frases, 1):
    print(f"\n--- Relato {i} ---")
    print(frase[:160] + ("..." if len(frase) > 160 else ""))
    print(sugerir_diagnosticos(frase, mapa))

## Observação (governança / limitações)

Este protótipo **não** substitui avaliação médica: é uma simulação educacional de apoio à decisão. Regras fixas e texto livre podem gerar **falsos positivos/negativos**; em produção seriam necessários validação clínica, dados representativos e transparência dos critérios.